# tanque-cheio · 02 — fine-tuning QLoRA (M6, M15)

Treina um adapter LoRA sobre o modelo base da config (`CONFIG`) com os pares de
`datasets/train.jsonl`, e depois gera as previsões do **dev2** com o adapter, pela mesma
lógica do `05_large_baselines.ipynb`. O teste nunca é usado aqui (regra 0.6).

- **Treino v2 (M15, padrão):** `CONFIG = "configs/train-v2.toml"`, sistema
  `ft2-qwen2.5-coder-1.5b-instruct`, mesmos hiperparâmetros da v1 (só muda o que
  `train-v2.toml` explica). Previsões no `datasets/dev2.jsonl` com `max_new_tokens = 384`
  (D31) e autocorreção com 1 nova tentativa (D32): grava `ft2-...__dev2.jsonl` e
  `ft2-...+retry__dev2.jsonl`, da mesma geração.
- **Refazer a v1 (M6):** `CONFIG = "configs/train.toml"`, `EVAL_DATASET =
  "datasets/dev.jsonl"`, `MAX_NEW_TOKENS = 256`, `RETRY = False` (sistema `ft-...`).
- Cada par vira prompt/completion: o prompt é o prompt único
  (`tanque.prompting.build_prompt`) pelo chat template do modelo, **idêntico ao da
  inferência**; a completion é a SQL. A perda é só na SQL (`completion_only_loss`).
- QLoRA: base em 4-bit nf4 (bitsandbytes), LoRA nas projeções de atenção e MLP, fp16
  (a T4 não tem bf16), gradient checkpointing. A lógica está em `tanque.train`
  (testada em CPU no repositório).
- Defesas do M6: uma GPU só (`CUDA_VISIBLE_DEVICES = "0"` antes do torch), `torchao`
  removido e `is_torchao_available()` checado antes do treino.

**No Kaggle:** Settings → Accelerator → **GPU T4**; Settings → **Internet on**. Depois
*Save Version → Save & Run All*. Tempo estimado do treino v2: ~5 h de GPU, mais ~10 min
de previsões (ver `results/training-v2.md`). Para medir o tempo antes, rode com
`MAX_STEPS = 30` e `EVAL_LIMIT = 3`.

**Saída** (`/kaggle/working/`): `adapter/` (adapter LoRA + model card provisório),
`train-log__<sistema>.json` (config, curvas de perda, tempo de GPU, versões),
`<sistema>__dev2.jsonl`, `<sistema>+retry__dev2.jsonl` e um `run-info__...json` para
cada um.

**Hugging Face (opcional):** só envia o adapter se `PUSH_TO_HUB = True`, `HF_REPO`
estiver preenchido e o notebook tiver o secret `HF_TOKEN` (Add-ons → Secrets, token
com permissão de escrita). Na v2, a Isabela publica pelo computador, depois de avaliar
no dev2 (`notebooks/README.md`, seção M15).

In [ ]:
# Parâmetros: edite só esta célula.
REPO_REF = "main"  # branch, tag ou commit do tanque-cheio
CONFIG = "configs/train-v2.toml"  # v2 (M15): sistema ft2-<modelo>; v1: "configs/train.toml"
BASE_MODEL = None  # None = o da config; ex.: "Qwen/Qwen3-1.7B" (ver train.toml)
MAX_TRAIN_EXAMPLES = None  # None = o da config (0 = todos); ex.: 4000
EPOCHS = None  # None = o da config
MAX_STEPS = None  # ex.: 30 para medir s/passo; None = rodada completa
EVAL_DATASET = "datasets/dev2.jsonl"  # v2: dev2 (D30); o teste nunca roda aqui
EVAL_LIMIT = None  # ex.: 3 para um teste rápido das previsões
MAX_NEW_TOKENS = 384  # D31 (a v1 usava 256)
RETRY = True  # D32: grava também <sistema>+retry__<split>.jsonl
HF_REPO = None  # v2: "belasoares/tanque-cheio-qwen2.5-coder-1.5b-lora-v2"
PUSH_TO_HUB = False  # True só com aprovação; precisa do secret HF_TOKEN
HF_PRIVATE = True  # o repo nasce privado; tornar público é um passo à parte
DEVICE = "cuda"
OUT_DIR = "/kaggle/working"  # vira a saída do notebook no Kaggle
REPO_DIR = "/tmp/tanque-cheio"  # fora de OUT_DIR, para não ir para a saída

In [ ]:
# Instala o pacote `tanque` no REPO_REF (com as versões de treino fixadas) e clona o
# repositório no mesmo ref, para ter datasets/, configs/ e data/manifest.json.
import os

# O Kaggle pode dar 2 GPUs T4; o SFTTrainer não aceita DataParallel. Uma GPU só,
# antes de qualquer import do torch.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import subprocess
import sys

REPO_URL = "https://github.com/belasoaress/tanque-cheio"
# Versões testadas no ensaio em CPU (uv.lock, grupos `infer` e `train`); torch vem da
# imagem do Kaggle.
PINS = [
    "transformers==5.21.0",
    "peft==0.21.2",
    "trl==1.16.0",
    "accelerate==1.15.0",
    "datasets==5.1.0",
    "bitsandbytes==0.50.2",
]


def sh(*cmd: str) -> None:
    print("$", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True)


sh(sys.executable, "-m", "pip", "install", "-q", f"tanque @ git+{REPO_URL}@{REPO_REF}", *PINS)
# A imagem do Kaggle traz um torchao antigo (0.10) e o peft se recusa a carregar o
# adapter quando ele está instalado. O projeto não usa torchao: remove.
sh(sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao")
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    sh("git", "clone", "--quiet", REPO_URL, REPO_DIR)
sh("git", "-C", REPO_DIR, "fetch", "--quiet", "origin", REPO_REF)
sh("git", "-C", REPO_DIR, "checkout", "--quiet", "FETCH_HEAD")
REPO_COMMIT = subprocess.check_output(
    ["git", "-C", REPO_DIR, "rev-parse", "HEAD"], text=True
).strip()
os.environ["TANQUE_ROOT"] = REPO_DIR  # antes de importar `tanque`
print("tanque-cheio em", REPO_COMMIT)

In [ ]:
import torch
from peft.import_utils import is_torchao_available

from tanque.train import finetune

print(finetune.versions())
if torch.cuda.is_available():
    print(
        "GPU:", torch.cuda.get_device_name(0), "| capability", torch.cuda.get_device_capability(0)
    )
else:
    print("Sem GPU: ligue o acelerador GPU T4 em Settings > Accelerator.")

# Falha cedo (antes do treino/inferência) se o peft não puder carregar adapters.
is_torchao_available()  # levanta ImportError se houver um torchao incompatível

In [ ]:
import json

from tanque import config
from tanque.eval.score import read_jsonl
from tanque.train import config as train_config

cfg = train_config.load_train_config(CONFIG)
overrides = {"model": {}, "data": {}, "training": {}}
if BASE_MODEL:
    overrides["model"]["base_model"] = BASE_MODEL
    overrides["model"]["revision"] = ""  # o commit fixado na config é do outro modelo
if MAX_TRAIN_EXAMPLES is not None:
    overrides["data"]["max_train_examples"] = MAX_TRAIN_EXAMPLES
if EPOCHS is not None:
    overrides["training"]["epochs"] = EPOCHS
cfg = train_config.with_overrides(cfg, overrides)
n_items = len(read_jsonl(config.resolve_path(cfg["data"]["train_path"])))
estimate = train_config.estimate_steps(cfg, n_items)
print(json.dumps(cfg, ensure_ascii=False, indent=2))
print("sistema:", train_config.system_name(cfg), "| pares no arquivo:", n_items)
print("batch efetivo:", train_config.effective_batch_size(cfg), "| estimativa:", estimate)

In [ ]:
# Banco DuckDB do snapshot versionado (D25), para a autocorreção executar as SQLs
# (~6 s, sem download). Equivale a `tanque data build --from-parquet ...`.
if RETRY:
    sh(
        sys.executable,
        "-c",
        "from tanque.cli import app; app()",
        "data",
        "build",
        "--from-parquet",
        os.path.join(REPO_DIR, "snapshot", "coletas-2026-09.parquet"),
    )

In [ ]:
info = finetune.run_finetune(cfg, out_dir=OUT_DIR, device=DEVICE, max_steps=MAX_STEPS)
projected_h = info["seconds_per_step"] * estimate["total_steps"] / 3600
print(
    f"Projeção da rodada completa ({estimate['total_steps']} passos): {projected_h:.1f} h de treino"
)

In [ ]:
# Curvas de perda (para colar em results/training.md).
print(finetune.loss_table(info["log_history"]))

In [ ]:
# Previsões no DEV2 com o adapter recém-treinado (mesma lógica do 05_large_baselines):
# gulosa, MAX_NEW_TOKENS, e com RETRY a 1 nova tentativa (D32) no mesmo laço.
from pathlib import Path

from tanque.infer import large, local

run = local.run_predictions(
    config.resolve_path(EVAL_DATASET),
    cfg["model"]["base_model"],
    out_dir=OUT_DIR,
    adapter_id=info["adapter_dir"],
    revision=cfg["model"]["revision"] or None,
    system=info["system"],
    limit=EVAL_LIMIT,
    dtype="float16",
    device=DEVICE,
    max_new_tokens=MAX_NEW_TOKENS,
    chat_template_kwargs=cfg["model"]["chat_template_kwargs"],
    system_suffix=cfg["model"]["system_suffix"],
    retry=RETRY,
    db_path=config.resolve_path("data/tanque.duckdb") if RETRY else None,
)
written = large.write_run_infos(
    run,
    OUT_DIR,
    repo_ref=REPO_REF,
    repo_commit=REPO_COMMIT,
    config_file=CONFIG,
    train_log=f"train-log__{info['system']}.json",
)
print("run-info:", [p.name for p in written])
for line in Path(run["path"]).read_text(encoding="utf-8").splitlines()[:3]:
    p = json.loads(line)
    print(p["id"], p["latency_ms"], "ms |", p["error"] or p["sql"][:200].replace("\n", " "))
if RETRY:
    print("nova tentativa:", run["retry_summary"])

In [ ]:
# Model card provisório no adapter (vai junto se o adapter for publicado).
card = finetune.model_card(
    {
        **info,
        "repo_commit": REPO_COMMIT,
        "config_file": CONFIG,
        "eval_split": Path(EVAL_DATASET).stem,
    }
)
Path(info["adapter_dir"], "README.md").write_text(card, encoding="utf-8")

# Publicação no Hugging Face: só com PUSH_TO_HUB = True, HF_REPO e o secret HF_TOKEN.
if PUSH_TO_HUB:
    if not HF_REPO:
        raise ValueError("PUSH_TO_HUB = True exige HF_REPO (ex.: 'usuario/tanque-cheio-...-lora')")
    from huggingface_hub import HfApi
    from kaggle_secrets import UserSecretsClient

    api = HfApi(token=UserSecretsClient().get_secret("HF_TOKEN"))
    api.create_repo(HF_REPO, private=HF_PRIVATE, exist_ok=True)
    commit = api.upload_folder(
        repo_id=HF_REPO,
        folder_path=info["adapter_dir"],
        commit_message=f"Adapter LoRA (tanque-cheio {REPO_COMMIT[:7]})",
    )
    print("Adapter publicado:", commit)
else:
    print("PUSH_TO_HUB = False: nada foi publicado.")